In [58]:
import numpy as np
from IPython.display import display, Audio
import soundfile as sf
from pydub import AudioSegment
import librosa
import torch, torchaudio

In [33]:
AUDIO_PATH = './sample.wav'

In [47]:
audio, sr = librosa.load(AUDIO_PATH, sr=None)
print(sr)
display(Audio(AUDIO_PATH))
sf.write("ori.wav", audio, sr)

44100


In [50]:
audio, sr = librosa.load('./ori.wav', sr=None)

In [35]:
pitch_up = librosa.effects.pitch_shift(audio, sr=sr, n_steps=5)

sf.write("pitched_up.wav", pitch_up, sr)

display(Audio('./pitched_up.wav'))

c:\Users\Hans\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
c:\Users\Hans\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(


In [36]:
white_noise = np.random.normal(0, audio.std(), len(audio))

whitenoiseaudio = audio + white_noise * 0.1
sf.write("white_noise.wav", whitenoiseaudio, sr)
display(Audio('./white_noise.wav'))

In [49]:
repeat = AudioSegment.from_wav('./ori.wav')
repeated_audio = repeat * 5
repeated_audio.export("repeated.wav", format="wav")

display(Audio('./repeated.wav'))

In [57]:
fade_in = AudioSegment.from_wav('./ori.wav')

fade_in = fade_in.fade_in(1000)
fade_in.export("fade_in.wav", format="wav")

display(Audio('./fade_in.wav'))

In [59]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [63]:
if sr != bundle.sample_rate:
    audio = torchaudio.functional.resample(audio, sr, bundle.sample_rate)

TypeError: arange() received an invalid combination of arguments - got (int, int, device=str, dtype=numpy.dtypes.Float32DType), but expected one of:
 * (Number end, *, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (Number start, Number end, *, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (Number start, Number end, Number step = 1, *, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [69]:
model = bundle.get_model()

In [70]:
def Transcriber(audiopath):
    audio, sr = torchaudio.load(audiopath)

    if sr != bundle.sample_rate:
        audio = torchaudio.functional.resample(audio, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(audio)

    class GreedyCTCDecoder (torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor):
            idx = torch.argmax(emission, dim=-1)
            idx = torch.unique_consecutive(idx, dim=-1)
            idx = [i for i in idx if i != self.blank]
            return "".join([self.labels[i] for i in idx])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    text = decoder(emission[0])
    text = text.replace("|", " ")
    return text

In [78]:
# def predict_text():
#     audio, sample_rate = torchaudio.load(AUDIO_PATH)

#     if sample_rate != bundle.sample_rate:
#         audio = torchaudio.functional.resample(audio, sample_rate, bundle.sample_rate)

#     with torch.inference_mode():
#         emission, _ = model(audio)

#     labels = bundle.get_labels()

#     indexs = torch.argmax(emission[0], dim=-1)
#     indexs = torch.unique_consecutive(indexs, dim=-1)
#     indexs = [i for i in indexs if i != 0]
    
#     prediction = "".join([labels[i] for i in indexs])

#     print("Prediction: ", prediction.replace("|", " "))

In [73]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H

In [94]:
def predict_text():
    audio, sr = torchaudio.load('ori.wav')

    if sr != bundle.sample_rate:
        audio = torchaudio.functional.resample(audio, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(audio)

    labels = bundle.get_labels()

    idx = torch.argmax(emission[0], dim=-1)
    idx = torch.unique_consecutive(idx, dim=-1)
    idx = [i for i in idx if i != 0]

    pred = "".join([labels[i] for i in idx])

    print(pred.replace("|", ' '))

    print(labels)
    print(idx)

In [95]:
predict_text()

I REALLY LOVE TITE SO I LIKE TO HUG TETE 
('-', '|', 'E', 'T', 'A', 'O', 'N', 'I', 'H', 'S', 'R', 'D', 'L', 'U', 'M', 'W', 'C', 'F', 'G', 'Y', 'P', 'B', 'V', 'K', "'", 'X', 'J', 'Q', 'Z')
[tensor(7), tensor(1), tensor(10), tensor(2), tensor(4), tensor(12), tensor(12), tensor(19), tensor(1), tensor(12), tensor(5), tensor(22), tensor(2), tensor(1), tensor(3), tensor(7), tensor(3), tensor(2), tensor(1), tensor(9), tensor(5), tensor(1), tensor(7), tensor(1), tensor(12), tensor(7), tensor(23), tensor(2), tensor(1), tensor(3), tensor(5), tensor(1), tensor(8), tensor(13), tensor(18), tensor(1), tensor(3), tensor(2), tensor(3), tensor(2), tensor(1)]


In [ ]:
def Transcriber(audio_path):
    audio, sr = torchaudio.load(audio_path)

    if sr != bundle.sample_rate:
        audio = torchaudio.functional.resample(audio, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(audio)

In [66]:
string = Transcriber('./ori.wav')

print(string)

I REALLY LOVE TITE SO I LIKE TO HUG TETE 


In [96]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [97]:
def predict():
    audio, sf = torchaudio.load('./ori.wav')

    if sf != bundle.sample_rate:
        audio = torchaudio.functional.resample(audio, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(audio)

    labels = bundle.get_labels()

    idx = torch.argmax(emission[0], dim=-1)
    idx = torch.unique_consecutive(idx, dim=-1)
    idx = [i for i in idx if i != 0]

    pred = "".join([labels[i] for i in idx])

    print(pred.replace("|", " "))

predict()

I REALLY LOVE TITE SO I LIKE TO HUG TETE 
